In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

from  sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score

from sklearn.preprocessing import KBinsDiscretizer
from sklearn.compose import ColumnTransformer


In [5]:
df = pd.read_csv("train.csv", usecols=['Age', 'Fare', 'Survived'])

In [6]:
df.head()

,Survived,Age,Fare
0,0,22.0,7.2500
1,1,38.0,71.2833
2,1,26.0,7.9250
3,1,35.0,53.1000
4,0,35.0,8.0500


In [7]:
df.dropna(inplace=True)

In [8]:
X = df.iloc[:,1:]
y = df.iloc[:,0]

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [10]:
X_train.head()

,Age,Fare
328,31.0,20.5250
73,26.0,14.4542
253,30.0,16.1000
719,33.0,7.7750
666,25.0,13.0000


In [15]:
clf = DecisionTreeClassifier()

In [16]:
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

In [17]:
accuracy_score(y_test, y_pred)

0.6363636363636364

In [19]:
np.mean(cross_val_score(DecisionTreeClassifier(),X,y,cv=10,scoring='accuracy'))

np.float64(0.6288928012519561)

In [34]:
kbin_age = KBinsDiscretizer(n_bins=15, encode='ordinal', strategy='quantile', quantile_method='averaged_inverted_cdf')
kbin_fare = KBinsDiscretizer(n_bins=15, encode='ordinal', strategy='quantile', quantile_method='averaged_inverted_cdf')

In [35]:
trf = ColumnTransformer([
    ('first',kbin_age,[0]),
    ('second',kbin_fare,[1])
])

In [36]:
X_train_trf = trf.fit_transform(X_train)
X_test_trf = trf.transform(X_test)

In [38]:
trf.named_transformers_['first'].bin_edges_

array([array([ 0.42,  6.  , 16.  , 19.  , 21.  , 23.  , 25.  , 28.  , 30.  ,
              32.  , 35.  , 38.  , 42.  , 47.  , 54.  , 80.  ])             ],
      dtype=object)

In [40]:
output = pd.DataFrame({
    'age':X_train['Age'],
    'age_trf':X_train_trf[:, 0].ravel(),
    'fare':X_train['Fare'],
    'fare_trf':X_train_trf[:, 1].ravel()
})

In [41]:
output['age_labels'] = pd.cut(x=X_train['Age'],
                              bins = trf.named_transformers_['first'].bin_edges_[0].tolist())
output['age_labels'] = pd.cut(x=X_train['Fare'],
                              bins = trf.named_transformers_['second'].bin_edges_[0].tolist())

                              

In [42]:
output.sample(5)

,age,age_trf,fare,fare_trf,age_labels
395,22.0,4.0,7.7958,2.0,"(7.775, 7.896]"
723,50.0,13.0,13.0000,6.0,"(10.5, 13.0]"
189,36.0,10.0,7.8958,3.0,"(7.775, 7.896]"
134,25.0,6.0,13.0000,6.0,"(10.5, 13.0]"
651,18.0,2.0,23.0000,8.0,"(18.75, 26.0]"
